<a href="https://colab.research.google.com/github/routparam12/Python_qns/blob/main/RAG-base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [55]:
!pip install pypdf faiss-cpu sentence-transformers google-generativeai -q

In [56]:
from google.colab import files

uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]

print(pdf_path)

Saving User_Churn_Report_Metric_Definitions_v3.pdf to User_Churn_Report_Metric_Definitions_v3 (2).pdf
User_Churn_Report_Metric_Definitions_v3 (2).pdf


In [57]:
from pypdf import PdfReader

reader = PdfReader(pdf_path)

text = ""

for page in reader.pages:
    text += page.extract_text() + "\n"

print(text[:1000])


User Churn Report
Metric Glossary & Definitions
Version 3 | Updated: October 2026 | Internal Reference Only
Change indicators:
 UPDATED
Definition updated
 RENAMED
Metric renamed
 NEW
New metric added
Panel Size & Acquisition
Total Panelists
Joined
 UPDATED
Number of panelists who joined the panel in the given month.
■ Previously 'Total Users' (cumulative). Now shows monthly new joins only.
DOI (Double Opt-In)
Count of newly joined panelists in the month whose email address has been
verified.
Joined & Attempted
Count of panelists who joined in a given month and made at least one survey
attempt within that same join month.
Survey Start Rate
Share of newly verified panelists who went on to attempt at least one survey in
their join month.
 Formula: Joined & Attempted / DOI x 100
Activity & Engagement
Total Active
(Monthly)
 RENAMED
Number of panelists who participated in at least one survey in both the current
month AND the immediately preceding month (consecutive month activity). A
panel

In [58]:
chunk_size = 500

chunks = [
    text[i:i+chunk_size]
    for i in range(0, len(text), chunk_size)
]

print("Total Chunks:", len(chunks))


Total Chunks: 8


In [59]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

embeddings = embedding_model.encode(chunks)

print(embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(8, 384)


In [60]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print("Vectors Added:", index.ntotal)

Vectors Added: 8


In [61]:
query = "How to calculate active last month not his month?"

In [62]:
# query_embedding = embedding_model.encode([query])

# distances, indices = index.search(
#     np.array(query_embedding),
#     k=3
# )

# retrieved_chunks = [chunks[i] for i in indices[0]]

# for i, chunk in enumerate(retrieved_chunks):
#     print(f"\nChunk {i+1}\n")
#     print(chunk[:500])

In [63]:
!pip install langchain-text-splitters -q

In [64]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configure a production-grade chunk splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # Max size of each chunk (characters)
    chunk_overlap=50,     # Overlapping characters between adjacent chunks
    length_function=len,
    separators=["\n\n", "\n", " ", ""] # Attempts to split on paragraphs first, then sentences, etc.
)

# Split your document
production_chunks = text_splitter.split_text(text)

print(f"Total production chunks: {len(production_chunks)}")
print("\n--- First Chunk Preview ---\n")
print(production_chunks[0])

Total production chunks: 10

--- First Chunk Preview ---

User Churn Report
Metric Glossary & Definitions
Version 3 | Updated: October 2026 | Internal Reference Only
Change indicators:
 UPDATED
Definition updated
 RENAMED
Metric renamed
 NEW
New metric added
Panel Size & Acquisition
Total Panelists
Joined
 UPDATED
Number of panelists who joined the panel in the given month.
■ Previously 'Total Users' (cumulative). Now shows monthly new joins only.
DOI (Double Opt-In)
Count of newly joined panelists in the month whose email address has been
verified.


In [65]:
import google.generativeai as genai
from google.colab import userdata

# Retrieve the API key securely from Colab Secrets
api_key = userdata.get("GEMINI_API_KEY")

genai.configure(
    api_key=api_key
)

model = genai.GenerativeModel(
    "gemini-2.5-flash"
)

In [66]:
context = "\n\n".join(retrieved_chunks)

prompt = f"""
Great first,(Say:Hi 0r  Hello)
Answer only from the provided context i natural language.

Context:
{context}

Question:
{query}

If answer is not present,
say "Not found in document".
"""

In [ ]:
response = model.generate_content(
    prompt
)

print(response.text)
